In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
sys.path.append(os.path.join(os.path.pardir, 'gotmtool'))
from gotmtool import *
from sbl_bbl import *

In [ ]:
la = np.sqrt(ustar/us0) 
print('La = {:6.3f}'.format(la))
Ti = inertial_period(lat)
print('Ti = {:6.3f}'.format(Ti))

In [ ]:
casename = 'lsc_ymc22_sbl_bbl_v2'
turbmethods = ['KPPLT-LF17', 'KPPLT-LF17-E', 'KPPLT-LF17-IC', 'KPPLT-LF17-E-IC']

res = 'L144'
trlx = 0
ds1_pfls = {}
ds2_pfls = {}
for turbmethod in turbmethods:
    gotm_dir = os.path.join(os.path.pardir, 'gotm', 'run', '{:s}'.format(casename))
    gotm_sim = Simulation(path=os.path.join(gotm_dir, '{:s}_{:s}_Rlx{:g}'.format(turbmethod, res, trlx)))
    ds1_pfls[turbmethod] = gotm_sim.load_data()
    gotm_dir = os.path.join(os.path.pardir, 'gotm', 'run', '{:s}_rf'.format(casename))
    gotm_sim = Simulation(path=os.path.join(gotm_dir, '{:s}_{:s}_Rlx{:g}'.format(turbmethod, res, trlx)))
    ds2_pfls[turbmethod] = gotm_sim.load_data()

In [ ]:
figpath  = 'overview_{:s}'.format(casename)
os.makedirs(figpath, exist_ok=True)

In [ ]:
def get_das(ds):
    das = dict(
            NN = ds.data_vars['temp'].differentiate(coord='z')*alphaT*g/N2,
            SS = ((ds.data_vars['u']+ds.data_vars['us']).differentiate(coord='z')**2+(ds.data_vars['v']+ds.data_vars['vs']).differentiate(coord='z')**2)/N2,
            # Ri = ds.data_vars['temp'].differentiate(coord='z')*alphaT*g/((ds.data_vars['u']+ds.data_vars['us']).differentiate(coord='z')**2+(ds.data_vars['v']+ds.data_vars['vs']).differentiate(coord='z')**2),
            Ri = (ds.data_vars['temp'].differentiate(coord='z')*alphaT*g-0.25*((ds.data_vars['u']+ds.data_vars['us']).differentiate(coord='z')**2+(ds.data_vars['v']+ds.data_vars['vs']).differentiate(coord='z')**2))/N2,       
            wb = get_flux(ds.data_vars['temp'].squeeze(),
                          ds.data_vars['nuh'].squeeze(),
                          ds.data_vars['gamh'].squeeze())*alphaT*g/ustar/bstar*1e3,
            wu = (get_flux(ds.data_vars['u'].squeeze(),
                          ds.data_vars['num'].squeeze(),
                          ds.data_vars['gamu'].squeeze())
                  -ds.data_vars['nucl'].squeeze()*ds.data_vars['dusdz'].squeeze())/ustar**2,
            wv = (get_flux(ds.data_vars['v'].squeeze(),
                          ds.data_vars['num'].squeeze(),
                          ds.data_vars['gamv'].squeeze())
                  -ds.data_vars['nucl'].squeeze()*ds.data_vars['dvsdz'].squeeze())/ustar**2,
        )
    return das

In [ ]:
edges1 = {}
edges2 = {}
for turbmethod in turbmethods:
    edges1[turbmethod], edges2[turbmethod] = [get_edges(nondim_da((ds.data_vars['temp'].differentiate(coord='z')*alphaT*g/N2)[:,:,0,0], H=H, Tf=Ti)) 
                                              for ds in [ds1_pfls[turbmethod], ds2_pfls[turbmethod]]]

In [ ]:
colors = {
    'KPPLT-LF17'       : 'tab:blue',
    'KPPLT-LF17-E'     : 'tab:red',
    'KPPLT-LF17-IC'    : 'tab:green',
    'KPPLT-LF17-E-IC'  : 'tab:orange',
}
labels = {
    'KPPLT-LF17'       : 'LF17',
    'KPPLT-LF17-E'     : 'LF17-E',
    'KPPLT-LF17-IC'    : 'LF17-IC',
    'KPPLT-LF17-E-IC'  : 'LF17-E-IC',
}
U0 = [0.25, -0.25]
abc = ['ad', 'be', 'cf']
tags = ['Aligned', 'Opposite']
tagy = ['bottom', 'top']
txx = ['left', 'right']
textx = {'left':0.1, 'right': 0.9}
texty = {'top': 0.92, 'bottom': 0.08}
xlabels = ['$(\overline{u}-U_0)/u_*$', '$\overline{w^\prime u^\prime}/u_*^2$']
line_kwargs=dict(linestyle='-', linewidth=1)
line_kwargs1 = dict(linestyle='--', linewidth=0.75)
line_kwargs2 = dict(linestyle='-', linewidth=0.75)
fig, axarr = plt.subplots(2, 3, sharey='row', gridspec_kw={'width_ratios': [3, 1, 1]})
fig.set_size_inches(8,5)
dss_pfls = [ds1_pfls, ds2_pfls]
for k, edges in enumerate([edges1, edges2]):
    ax = axarr[k,0]
    for turbmethod in turbmethods:
        edges[turbmethod][0].rolling(time=5, center=True).mean().plot(ax=ax, color=colors[turbmethod], **line_kwargs, label=labels[turbmethod])
        edges[turbmethod][1].rolling(time=5, center=True).mean().plot(ax=ax, color=colors[turbmethod], **line_kwargs)
    ax.set_ylabel('$z/H$')
    ax.set_ylim([-1, 0])
    ax.set_xlim([0, 12])
    ax.text(0.05, texty['top'], '({:s})'.format(abc[0][k]), transform=ax.transAxes, va='top', ha='left')
    ax.text(0.95, texty[tagy[k]], tags[k], transform=ax.transAxes, va=tagy[k], ha='right')
    if k == 0:
        ax.legend(loc='upper right', ncol=2, fontsize=9)
        ax.set_xlabel('')
    else:
        ax.set_xlabel('$t/T_f$')
    
    for turbmethod in turbmethods:
        time_merge = edges[turbmethod][0].dropna(dim='time').time[-1]
        tslice1 = slice(2, 3)
        tslice2 = slice(time_merge + 4, time_merge + 5)
        ds = dss_pfls[k][turbmethod]
        da = nondim_da((ds.data_vars['u']+ds.data_vars['us']-U0[k]).squeeze()/ustar, H=H, Tf=Ti)
        da.sel(time=tslice1).mean(dim='time').plot(ax=axarr[k,1], y=da.dims[0], color=colors[turbmethod], **line_kwargs1)
        da.sel(time=tslice2).mean(dim='time').plot(ax=axarr[k,1], y=da.dims[0], color=colors[turbmethod], **line_kwargs2)
        da = nondim_da((get_flux(ds.data_vars['u'].squeeze(),
                       ds.data_vars['num'].squeeze(),
                       ds.data_vars['gamu'].squeeze())
              -ds.data_vars['nucl'].squeeze()*ds.data_vars['dusdz'].squeeze())/ustar**2, H=H, Tf=Ti)
        da.sel(time=tslice1).mean(dim='time').plot(ax=axarr[k,2], y=da.dims[0], color=colors[turbmethod], **line_kwargs1)
        da.sel(time=tslice2).mean(dim='time').plot(ax=axarr[k,2], y=da.dims[0], color=colors[turbmethod], **line_kwargs2)

    for i in np.arange(2):
        ax = axarr[k,i+1]
        ax.set_title('')
        ax.set_ylabel('')
        ax.text(textx[txx[k]], texty['top'], '({:s})'.format(abc[i+1][k]), transform=ax.transAxes, va='top', ha=txx[k])
        if k == 0:
            ax.set_xlabel('')
        else:
            ax.set_xlabel(xlabels[i])
        ax.axvline(x=0, linewidth=0.75, color='k', zorder=0)
l1, = plt.plot(np.nan,np.nan,'k',**line_kwargs1)
l2, = plt.plot(np.nan,np.nan,'k',**line_kwargs2)
axarr[1,1].legend([l1,l2],['T1','T2'], loc='center right', fontsize=9)

plt.tight_layout()
plt.subplots_adjust(wspace=0.12)
figname = os.path.join(figpath, 'IC-sensitivity_gotm-v2-LF17')
fig.savefig(figname, dpi = 300, facecolor='w')